## Configuration


In [ ]:
dbutils.widgets.text("catalog", "training_lab")
dbutils.widgets.text("schema", "student_01")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", schema)
ns = f"{catalog}.{schema}"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
print("Working namespace:", ns)
from pyspark.sql import functions as F


## Inspect and chart


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pdf = spark.table(ns+".day18_daily_demand").orderBy("ds").toPandas()
pdf["ds"] = pd.to_datetime(pdf["ds"])
pdf = pdf.sort_values("ds").reset_index(drop=True)
assert len(pdf)==140 and pdf.ds.is_unique and pdf.y.notna().all()
assert (pdf.ds.diff().dropna() == pd.Timedelta(days=1)).all()
series = pdf.set_index("ds")["y"].asfreq("D")
fig, ax = plt.subplots(figsize=(9,3))
series.plot(ax=ax, label="Daily units")
series.rolling(7, min_periods=7).mean().plot(ax=ax,label="7 day mean")
ax.set(ylabel="Units", xlabel="Date")
ax.legend(); display(fig); plt.close(fig)


## Missing date example


In [ ]:
# Independent toy example: a missing date means unknown.
toy = pd.DataFrame({"ds":pd.to_datetime(["2026-01-01","2026-01-03"]),
                    "y":[100.0,140.0]})
calendar = pd.date_range("2026-01-01","2026-01-03",freq="D")
aligned = toy.set_index("ds").reindex(calendar)
print(aligned)  # January 2 is NaN, not automatically zero.
# For a confirmed closed shop only:
confirmed_closed = aligned.copy()
confirmed_closed.loc[pd.Timestamp("2026-01-02"),"y"] = 0
print(confirmed_closed)


## Split and decompose


In [ ]:
from statsmodels.tsa.seasonal import STL
train = series.iloc[:112]
valid = series.iloc[112:126]
test = series.iloc[126:]
assert train.index.max() < valid.index.min() < test.index.min()
parts = STL(train, period=7, robust=True).fit()
fig = parts.plot(); display(fig); plt.close(fig)
print("Split sizes:", len(train), len(valid), len(test))
